# [LEC] RAG 이론 & 실습

슬라이드와 설명, 퀴즈로 RAG(Retrieval Augmented Generation)의 개념과 핵심 모듈을 익힙니다.

In [ ]:
%cd ../../../course/Chapter01/notebooks

**목차**

- 1. Retrieval Augmented Generation (슬라이드) (퀴즈)
  - 1.1 Fine-Tuning과의 비교 (슬라이드) (퀴즈)
  - 1.2 Naive → Advanced → Modular RAG 발전 과정 (슬라이드) (퀴즈)
- 2. RAG 핵심 모듈 (슬라이드) (퀴즈)
  - 2.1 Loading, Splitting, Embedding (슬라이드) (퀴즈)
  - 2.2 Storage, Retrieval, Generation (슬라이드) (퀴즈)
- 3. 청킹 전략과 임베딩 (슬라이드) (퀴즈)
  - 3.1 Fixed-size, Recursive, Semantic Chunking (슬라이드) (퀴즈)
  - 3.2 임베딩 모델 선택 기준 (슬라이드) (퀴즈)
- 4. 정리 퀴즈 (퀴즈)

In [ ]:
import sys

sys.path.insert(0, "..")  # 챕터 폴더의 assets 도구를 불러오기 위한 경로

from assets.widgets import show_quiz, show_slides, slide_setup

slide_setup()  # 슬라이드 뷰어(PDF.js)를 불러옵니다. 슬라이드가 비어 보이면 이 셀을 다시 실행합니다.

In [ ]:
show_slides("1-4", [1, 2])

이번 시간은 RAG를 세 덩어리로 나눠 봅니다.

1. Retrieval Augmented Generation: RAG가 무엇이고 Fine-Tuning과 무엇이 다른지, Naive → Advanced → Modular로 어떻게 발전했는지
2. RAG 핵심 모듈: Loading, Splitting, Embedding, Storage, Retrieval, Generation 여섯 단계
3. 청킹 전략과 임베딩: 문서를 나누는 세 가지 방법과 임베딩 모델을 고르는 기준

이 노트북은 개념만 다룹니다. 슬라이드 속 코드는 읽기만 합니다.

<details><summary>강사용 메모</summary>

| 항목 | 내용 |
|---|---|
| 버전 | C |
| 원본 PDF | `docs/raw/Chapter01/day01/1-4-rag-theory-and-core-modules.pdf` (19쪽) |
| 커리큘럼 | 2.1 RAG 이론과 핵심 모듈 실습(2일차 3.0H). PDF는 `day01` 폴더에 있지만 커리큘럼상 2일차 첫 블록입니다. |
| 원본 코드 | 이 노트북에는 없습니다. `src/raw/Chapter01/03-rag/`의 다섯 파일은 `1-4_rag_LAB.ipynb`에서 다룹니다. |
| API 호출 | 없습니다. 슬라이드 준비 셀만 실행하면 됩니다. |

</details>

## 1. Retrieval Augmented Generation <small>(슬라이드) (퀴즈)</small>

### 1.1 Fine-Tuning과의 비교 <small>(슬라이드) (퀴즈)</small>

In [ ]:
show_slides("1-4", [3, 4])

RAG(Retrieval Augmented Generation, 검색 증강 생성)는 모델이 모르는 외부 지식을 먼저 찾아서, 답변을 만들 때 함께 넣어 주는 방식입니다.
LLM은 학습한 시점 이후의 정보나 사내 문서를 기본적으로 알지 못합니다.
그래서 질문과 관련 있는 문서를 먼저 찾고, 그 문서를 근거로 답변하게 합니다.
모델을 다시 학습시키지 않아도, 찾아볼 문서만 바꾸면 모델이 참고하는 지식이 바뀝니다.

슬라이드 아래쪽의 프롬프트가 RAG의 기본 모양입니다. `{question}` 자리에 사용자 질문이, `{context}` 자리에 찾아온 문서 내용이 들어갑니다.
"맥락에 답이 없으면 '문서에서 확인할 수 없습니다'라고 말하세요"처럼, 근거 밖의 내용을 지어내지 않게 하는 규칙도 함께 넣습니다.

**왜 배우나:** 업무에 쓰는 Agent는 모델이 학습하지 않은 사내 문서를 다뤄야 합니다. RAG는 그 문서를 모델에게 보여 주는 가장 기본적인 방법입니다.

In [ ]:
show_slides("1-4", [5])

Fine-Tuning(미세 조정)과 RAG는 풀려는 문제가 다릅니다.

- **Fine-Tuning**은 모델을 추가로 학습시켜 말투, 출력 형식, 업무 스타일을 맞추는 데 유리합니다.
- **RAG**는 모델은 그대로 두고 질문할 때 맥락을 더해, 최신 정보·사내 문서·제품 매뉴얼을 참조하는 데 유리합니다.

실제 서비스에서는 기본 모델, 프롬프트, RAG, Fine-Tuning을 목적에 맞게 조합해 씁니다.
한 줄로 줄이면 "Fine-Tuning은 모델을 학습시키고, RAG는 맥락을 추가한다"입니다.

In [ ]:
show_quiz("../assets/quizzes/1-4/q11.json")

**정리:** 말투·형식을 바꾸려면 Fine-Tuning, 모델이 모르는 지식을 쓰게 하려면 RAG입니다.

<details><summary>강사용 메모</summary>

- 슬라이드 프롬프트의 "맥락에 답이 없으면 …" 규칙은 실습 4장(RAG 체인)의 system 프롬프트에 더해 보는 CHANGE_HERE로 이어집니다. 원본 실습 코드의 system 문장에는 이 규칙이 없습니다.

</details>

### 1.2 Naive → Advanced → Modular RAG 발전 과정 <small>(슬라이드) (퀴즈)</small>

In [ ]:
show_slides("1-4", [6])

Naive RAG는 RAG의 가장 기본적인 형태이고, 다섯 단계로 움직입니다.

1. 문서를 불러옵니다.
2. 문서를 작은 청크(chunk: 문서를 나눈 조각)로 나눕니다.
3. 청크를 임베딩(글의 의미를 숫자 목록으로 바꾸는 일)해서 저장합니다.
4. 질문과 가까운 청크를 검색합니다.
5. 검색 결과를 프롬프트에 넣고 답변을 생성합니다.

1~3단계는 질문이 오기 전에 미리 해 두는 준비이고, 4~5단계는 질문이 올 때마다 합니다.
이 다섯 단계가 2장에서 볼 핵심 모듈의 뼈대입니다.

In [ ]:
show_slides("1-4", [7])

Advanced RAG는 Naive RAG에 검색 품질과 답변 품질을 높이는 보완 단계를 더한 형태입니다.

- **Query Rewrite**: 사용자의 질문을 검색하기 좋은 형태로 다시 씁니다.
- **Hybrid Search**: 키워드 검색과 벡터 검색을 함께 씁니다.
- **Re-ranking**: 검색 결과의 우선순위를 다시 계산합니다.
- **Self-check**: 답변과 근거가 맞는지 점검합니다.

In [ ]:
show_quiz("../assets/quizzes/1-4/q12.json")

In [ ]:
show_slides("1-4", [8])

Modular RAG는 RAG의 단계를 독립된 모듈처럼 떼었다 붙였다 조합하는 방식입니다.
슬라이드처럼 Query Rewrite, Retrieval, Re-ranking, Generation, Self-check를 각각 하나의 모듈로 보고 필요한 것을 골라 잇습니다.

**정리:** Naive RAG는 기본 다섯 단계, Advanced RAG는 여기에 보완 단계를 더한 형태, Modular RAG는 단계를 모듈처럼 조합하는 형태입니다.

<details><summary>강사용 메모</summary>

- Hybrid Search는 실습 6장 RAGaaS의 `retrieve-rrf`와 이어집니다. 원본 코드 주석에 "bm25(키워드), knn(의미), rrf(하이브리드) 중 선택"이라고 적혀 있습니다.

</details>

## 2. RAG 핵심 모듈 <small>(슬라이드) (퀴즈)</small>

### 2.1 Loading, Splitting, Embedding <small>(슬라이드) (퀴즈)</small>

In [ ]:
show_slides("1-4", [9, 10])

Loading(불러오기)은 원천 데이터를 RAG가 다룰 수 있는 문서 형태로 불러오는 단계입니다.
PDF, 웹 페이지, 텍스트 파일, 사내 문서 등을 읽어 들입니다.
이때 본문뿐 아니라 파일명, 페이지 번호, URL 같은 메타데이터(데이터에 대한 정보)도 함께 남기는 것이 중요합니다.

슬라이드 코드의 `PDFMinerLoader`는 PDF를 읽어 `Document` 목록으로 돌려줍니다. `Document` 하나에는 본문(`page_content`)과 메타데이터(`metadata`)가 들어 있습니다.

**왜 배우나:** 메타데이터가 있어야 답변에 출처를 붙이거나, 원하는 문서만 골라 검색할 수 있습니다.

In [ ]:
show_quiz("../assets/quizzes/1-4/q21.json")

In [ ]:
show_slides("1-4", [11])

Splitting(나누기)은 긴 문서를 검색하기 좋은 작은 단위로 나누는 단계입니다.
문서 전체를 한 번에 검색 대상으로 쓰기에는 너무 큽니다.
너무 크게 나누면 필요한 부분만 찾기 어렵고, 너무 작게 나누면 의미가 끊겨 답변에 필요한 맥락이 모자랄 수 있습니다.

슬라이드 코드에서 `chunk_size`는 청크 하나의 최대 길이, `chunk_overlap`은 이웃한 청크끼리 겹쳐 두는 길이입니다.
겹쳐 두면 경계에서 문장이 잘려도 앞뒤 맥락이 조금 남습니다. 대신 겹친 만큼 중복과 비용이 늘어납니다.

In [ ]:
show_slides("1-4", [12])

Embedding(임베딩)은 텍스트의 의미를 벡터(숫자 목록)로 바꾸는 단계입니다.
의미가 비슷한 문장은 벡터 공간에서도 가까운 곳에 놓이도록 바꿉니다.
질문도 같은 방식으로 벡터로 바꾼 뒤, 그 벡터와 가까운 문서 청크를 찾습니다.

슬라이드 코드의 `embed_query()`는 문장 하나를 벡터로 바꾸고, `len(vector)`는 그 벡터의 길이(차원 수)입니다.

**왜 배우나:** RAG의 검색 품질은 임베딩 모델의 품질에 크게 좌우됩니다. 그래서 3.2에서 임베딩 모델을 고르는 기준을 따로 봅니다.

**정리:** Loading은 본문과 메타데이터를 불러오고, Splitting은 검색하기 좋은 크기로 나누고, Embedding은 의미를 벡터로 바꿉니다.

<details><summary>강사용 메모</summary>

- 실습의 `PDFMinerLoader(경로)` 기본값은 약관 PDF 28쪽 전체를 `Document` 1개로 읽습니다. `metadata`에 `source`(파일 경로)와 `total_pages`는 있지만 쪽마다의 페이지 번호는 없습니다(사외 실행으로 확인). 슬라이드의 "페이지 번호 메타데이터"와 다르니, 물으면 로더 설정에 따라 달라진다고 답합니다. 실습 코드는 바꾸지 않았습니다.

---

- 슬라이드 코드는 `chunk_size=800`, `chunk_overlap=120`이고, 실습 코드는 `chunk_size=500`, `chunk_overlap=75`입니다. 물으면 "값은 문서에 맞게 정한다"고 답합니다.
- "겹친 만큼 중복과 비용이 늘어난다"는 원본 실습 코드 주석("클수록 문맥 연결성이 좋아지지만 중복과 비용이 늘어난다")에서 가져왔습니다.

---

- 슬라이드 코드는 `OpenAIEmbeddings(model="text-embedding-3-small")`입니다. 실습 코드는 `get_embeddings()`로 환경에 따라 사내 임베딩 또는 OpenAI 임베딩을 씁니다.
- 사내 임베딩 모델의 이름과 벡터 차원 수는 미확인입니다.

</details>

### 2.2 Storage, Retrieval, Generation <small>(슬라이드) (퀴즈)</small>

In [ ]:
show_slides("1-4", [13])

Storage(저장)는 임베딩한 청크를 원문, 메타데이터와 함께 저장하는 단계입니다.
Chroma, FAISS 같은 벡터 저장소를 씁니다.
벡터 저장소는 벡터 검색을 빠르게 하기 위한 인덱스(찾아보기 목록)를 관리합니다.
문서가 바뀌면 기존 인덱스를 어떻게 갱신할지도 함께 생각해야 합니다.

슬라이드 코드의 `persist_directory`는 저장소를 파일로 남길 폴더입니다. 이 값을 주지 않으면 저장소는 메모리에만 있다가 프로그램이 끝나면 사라집니다.

In [ ]:
show_slides("1-4", [14])

Retrieval(검색)은 사용자의 질문과 관련 있는 문서 청크를 찾는 단계입니다.
질문을 임베딩한 뒤, 벡터 저장소에서 가까운 청크를 찾습니다.
검색 개수, 유사도 기준, 메타데이터 필터가 결과 품질에 영향을 줍니다. 슬라이드 코드의 `search_kwargs={"k": 3}`는 가까운 청크 3개를 가져오라는 뜻입니다.

**왜 배우나:** 검색 결과가 틀리면 생성 단계가 좋아도 답변이 흔들립니다. RAG의 답변이 이상할 때는 검색해 온 청크부터 확인합니다.

In [ ]:
show_quiz("../assets/quizzes/1-4/q22.json")

In [ ]:
show_slides("1-4", [15])

Generation(생성)은 검색한 근거를 바탕으로 최종 답변을 만드는 단계입니다.
모델에게 질문과 검색 결과를 함께 주고, 검색된 근거 안에서 답변하도록 지시합니다.
필요하면 답변과 함께 출처도 표시합니다. 출처는 Loading 단계에서 남긴 메타데이터에서 가져옵니다.

**정리:** Storage는 벡터와 원문을 저장하고, Retrieval은 질문과 가까운 청크를 찾고, Generation은 그 근거 안에서 답변합니다.

<details><summary>강사용 메모</summary>

- 실습 코드는 `persist_directory` 없이 Chroma를 메모리에만 만듭니다. 실행해도 파일이 생기지 않습니다.

</details>

## 3. 청킹 전략과 임베딩 <small>(슬라이드) (퀴즈)</small>

### 3.1 Fixed-size, Recursive, Semantic Chunking <small>(슬라이드) (퀴즈)</small>

In [ ]:
show_slides("1-4", [16, 17])

청킹(chunking)은 문서를 청크로 나누는 방법입니다. 슬라이드는 세 가지를 비교합니다. 먼저 두 가지입니다.

- **Fixed-size Chunking**: 글자 수나 토큰 수를 기준으로 일정하게 나눕니다. 구현이 단순하고 결과를 예측하기 쉽지만, 문장이나 문단 중간에서 잘릴 수 있습니다. 문서 구조가 단순할 때 빠르게 실험하기 좋습니다.
- **Recursive Chunking**: 문단, 문장, 구분자 순서로 자연스러운 경계를 찾아 나눕니다. 의미 단위가 비교적 잘 보존되어 일반적인 문서에서 기본 선택지로 많이 씁니다. 문서 구조가 정리되어 있을수록 효과가 좋습니다.

2.1 슬라이드 코드의 `RecursiveCharacterTextSplitter`가 Recursive Chunking입니다.

In [ ]:
show_slides("1-4", [18])

- **Semantic Chunking**: 의미가 바뀌는 지점, 곧 주제가 바뀌는 곳을 기준으로 나눕니다. 보고서나 정책 문서처럼 의미 흐름이 중요한 자료에 알맞습니다. 대신 구현과 계산 비용이 상대적으로 커질 수 있습니다.

In [ ]:
show_quiz("../assets/quizzes/1-4/q31.json")

**정리:** Fixed-size는 단순하고, Recursive는 기본 선택지이고, Semantic은 의미 흐름을 지키는 대신 비용이 큽니다.

### 3.2 임베딩 모델 선택 기준 <small>(슬라이드) (퀴즈)</small>

In [ ]:
show_slides("1-4", [19])

임베딩 모델은 문장의 의미를 벡터로 얼마나 잘 나타내는지를 결정합니다. 고를 때는 다음을 봅니다.

- 한국어 문서라면 한국어 의미 유사도를 잘 반영하는 모델이 필요합니다.
- 도메인 용어가 많으면 실제 데이터로 검색 품질을 확인해야 합니다.
- 벡터 차원 수, 속도, 비용도 함께 고려합니다.

임베딩 모델 선택은 벤치마크 점수만으로 끝나지 않습니다. 실제 사용자 질문 샘플, 기대하는 정답 문서, 청킹 전략별 검색 결과, 비용과 응답 시간을 함께 확인합니다.

**왜 배우나:** 사내 문서에는 도메인 용어가 많습니다. 그래서 점수표보다 실제 질문으로 검색해 본 결과가 모델을 고르는 근거가 됩니다.

In [ ]:
show_quiz("../assets/quizzes/1-4/q32.json")

**정리:** 임베딩 모델은 한국어·도메인 용어·차원 수·속도·비용을 보고, 실제 질문으로 검색해 보고 고릅니다.

## 4. 정리 퀴즈 <small>(퀴즈)</small>

In [ ]:
show_quiz("../assets/quizzes/1-4/final.json")

**이 노트북에서 본 것**
- RAG는 모델이 모르는 외부 지식을 찾아 답변에 함께 넣는 방식입니다. Fine-Tuning은 모델을 학습시키고, RAG는 맥락을 추가합니다.
- Naive RAG는 불러오기 → 나누기 → 임베딩·저장 → 검색 → 생성의 다섯 단계이고, Advanced RAG는 Query Rewrite·Hybrid Search·Re-ranking·Self-check를 더하고, Modular RAG는 단계를 모듈처럼 조합합니다.
- 핵심 모듈은 Loading, Splitting, Embedding, Storage, Retrieval, Generation입니다. 메타데이터가 있어야 출처를 붙이고, 검색이 틀리면 답변도 흔들립니다.
- 청킹은 Fixed-size, Recursive, Semantic 세 가지가 있고, Recursive가 기본 선택지로 많이 쓰입니다.
- 임베딩 모델은 벤치마크 점수만 보지 않고, 실제 질문으로 검색해 보고 고릅니다.